In [42]:
import torch

In [43]:
img = torch.tensor(
    [
        [
            [
                [-1],[1],[2]
            ],
            [
                [3],[4],[5]
            ],
            [
                [6],[7],[8]
            ]
        ]
    ]
)

In [44]:
# def chunking(tensor, chunk_height, chunk_width, pad=0):
#     batch, height, width, channels = tensor.shape
#     num_h_steps = (height + chunk_height - 1) // chunk_height
#     height_pad = num_h_steps * chunk_height
#     num_w_steps = (width + chunk_width - 1) // chunk_width
#     width_pad = num_w_steps * chunk_width
#     height_needed_pad = height_pad - height
#     width_needed_pad = width_pad - width
#     num_chunks = num_h_steps * num_w_steps * channels
#     padding = (0, 0, 0, width_needed_pad, 0, height_needed_pad)
#     padded = torch.nn.functional.pad(tensor, padding, mode="constant", value=pad)
#     reshaped = padded.reshape(batch, num_h_steps, chunk_height, num_w_steps, chunk_width, channels)
#     transposed = reshaped.permute(0, 3, 1, 5, 2, 4)
#     output = transposed.reshape(batch, num_chunks, chunk_height * chunk_width)
#     return output
def chunking(tensor: torch.Tensor, chunk_height: int, chunk_width: int, pad: int = 0) -> torch.Tensor:
    """
    Reshapes a 4D tensor (batch, height, width, channels) into chunks.

    Args:
        tensor (torch.Tensor): Input tensor of shape (batch, height, width, channels).
        chunk_height (int): The height of each chunk.
        chunk_width (int): The width of each chunk.
        pad (int): Value used for padding if the dimensions are not perfectly divisible
                   by chunk_height/chunk_width. Defaults to 0.

    Returns:
        torch.Tensor: A 3D tensor of shape (batch, num_total_chunks, chunk_height * chunk_width).
                      Chunks are ordered by width_step, then height_step, then channel.
    """
    batch, height, width, channels = tensor.shape

    if chunk_height <= 0 or chunk_width <= 0:
        raise ValueError("chunk_height and chunk_width must be positive.")

    num_h_steps = (height + chunk_height - 1) // chunk_height
    height_padded = num_h_steps * chunk_height
    num_w_steps = (width + chunk_width - 1) // chunk_width
    width_padded = num_w_steps * chunk_width

    height_needed_pad = height_padded - height
    width_needed_pad = width_padded - width

    # Padding format for torch.nn.functional.pad is (pad_left_dimN, pad_right_dimN, ...).
    # For a 4D tensor (B, H, W, C), tensor.pad expects padding for last 3 dims (C, W, H)
    # So, padding tuple is (pad_C_left, pad_C_right, pad_W_left, pad_W_right, pad_H_left, pad_H_right)
    padding = (0, 0,  # Channels padding (none)
               0, width_needed_pad,   # Width padding (only on the right)
               0, height_needed_pad)  # Height padding (only at the bottom)

    padded_tensor = torch.nn.functional.pad(tensor, padding, mode="constant", value=pad)

    # Reshape to (batch, num_h_steps, chunk_height, num_w_steps, chunk_width, channels)
    reshaped = padded_tensor.reshape(batch, num_h_steps, chunk_height, num_w_steps, chunk_width, channels)

    # Permute to (batch, num_w_steps, num_h_steps, channels, chunk_height, chunk_width)
    # This order ensures that when we flatten, chunks are grouped by w_step, then h_step, then channel.
    # Original indices: (0:B, 1:N_h, 2:C_h, 3:N_w, 4:C_w, 5:Chan)
    # New order:        (0:B, 3:N_w, 1:N_h, 5:Chan, 2:C_h, 4:C_w)
    permuted = reshaped.permute(0, 3, 1, 5, 2, 4)

    # Reshape to (batch, num_total_chunks, chunk_area)
    # num_total_chunks = num_w_steps * num_h_steps * channels
    # chunk_area = chunk_height * chunk_width
    output_tensor = permuted.reshape(batch, num_w_steps * num_h_steps * channels, chunk_height * chunk_width)

    return output_tensor

## 1 by 1

In [45]:
chunk_height = 1
chunk_width = 1

In [46]:
chunked = chunking(img, chunk_height, chunk_width)

In [47]:
expected = torch.tensor(
    [
        [
            [-1],
            [3],
            [6],
            [1],
            [4],
            [7],
            [2],
            [5],
            [8]
        ]
    ]
)

In [48]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.


## 2 x 2

In [49]:
chunk_height = 2
chunk_width = 2

In [50]:
chunked = chunking(img, chunk_height, chunk_width)

In [51]:
expected = torch.tensor(
    [
        [
            [-1, 1, 3, 4],
            [6, 7, 0, 0],
            [2, 0, 5, 0],
            [8, 0, 0, 0]
        ]
    ]
)

In [52]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.


## 1 x 2

In [53]:
chunk_height = 1
chunk_width = 2

In [54]:
chunked = chunking(img, chunk_height, chunk_width)

In [55]:
expected = torch.tensor(
    [
        [
            [ -1, 1],
            [3, 4],
            [6, 7],
            [2, 0],
            [5, 0],
            [8, 0]
        ]
    ]
)

In [56]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.


## 2 x 1

In [57]:
chunk_height = 2
chunk_width = 1

In [58]:
chunked = chunking(img, chunk_height, chunk_width)

In [59]:
expected = torch.tensor(
    [
        [
            [ -1, 3],
            [6, 0],
            [1, 4],
            [7, 0],
            [2, 5],
            [8, 0]
        ]
    ]
)

In [60]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.


## Bigger image

In [61]:
img = torch.tensor(
    [
        [
            [
                [-1],[1],[2], [10]
            ],
            [
                [3],[4],[5], [11]
            ],
            [
                [6],[7],[8], [12]
            ]
        ]
    ]
)

## 2 x 2

In [62]:
chunk_height = 2
chunk_width = 2

In [63]:
chunked = chunking(img, chunk_height, chunk_width)

In [64]:
expected = torch.tensor(
    [
        [
            [-1, 1, 3, 4],
            [6, 7, 0, 0],
            [2, 10, 5, 11],
            [8, 12, 0, 0]
        ]
    ]
)

In [65]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.


## multi channel

In [66]:
img = torch.tensor(
    [
        [
            [
                [-1, -11, -1],[1, 1, 1],[2, 2, 2]
            ],
            [
                [3, 3, 3],[4, 4, 4],[5, 5, 5]
            ],
            [
                [6, 6, 6],[7, 7, 17],[8, 8, 8]
            ]
        ]
    ]
)

## 2 x 2

In [67]:
chunk_height = 2
chunk_width = 2

In [68]:
chunked = chunking(img, chunk_height, chunk_width)

In [69]:
expected = torch.tensor(
    [
        [
            [-1, 1, 3, 4],
            [-11, 1, 3, 4],
            [-1, 1, 3, 4],
            [6, 7, 0, 0],
            [6, 7, 0, 0],
            [6, 17, 0, 0],
            [2, 0, 5, 0],
            [2, 0, 5, 0],
            [2, 0, 5, 0],
            [8, 0, 0, 0],
            [8, 0, 0, 0],
            [8, 0, 0, 0]
        ]
    ]
)

In [70]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.


## multi channel and batched

In [71]:
img = torch.tensor(
    [
        [
            [
                [-1, -11, -1],[1, 1, 1],[2, 2, 2]
            ],
            [
                [3, 3, 3],[4, 4, 4],[5, 5, 5]
            ],
            [
                [6, 6, 6],[7, 7, 17],[8, 8, 8]
            ]
        ]
    ]
)

In [72]:
image = img[0]

In [73]:
img = torch.stack(
    [
        image,
        image,
        image
    ]
)

In [74]:
chunk_height = 2
chunk_width = 2

In [75]:
chunked = chunking(img, chunk_height, chunk_width)

In [76]:
expected = torch.tensor(
    [
        [
            [-1, 1, 3, 4],
            [-11, 1, 3, 4],
            [-1, 1, 3, 4],
            [6, 7, 0, 0],
            [6, 7, 0, 0],
            [6, 17, 0, 0],
            [2, 0, 5, 0],
            [2, 0, 5, 0],
            [2, 0, 5, 0],
            [8, 0, 0, 0],
            [8, 0, 0, 0],
            [8, 0, 0, 0]
        ]
    ]
)

In [77]:
expected = expected.repeat(3, 1, 1)

In [78]:
torch.testing.assert_close(chunked, expected)
print("Chunking test passed.")

Chunking test passed.
